In [13]:
from math import gcd
from math import isqrt
import itertools
from sympy import divisors

In [14]:
def search_solutions(a, b, c, d, Sj, Sj2, n):
    """
    Returns all solutions (a_i,b_i,c_i,d_i,k_i)
    satisfying the reduced system.
    """

    # Necessary condition
    if a * c * Sj != b * d * Sj2:
        return []

    g = gcd(b, c)
    bp = b // g
    cp = c // g

    rhs = a * b * Sj

    if rhs % (bp * bp) != 0:
        return []

    M = rhs // (bp * bp)

    solutions = []

    def build_solution(ts, ks):

        a_list = []
        b_list = []
        c_list = []
        d_list = []

        for t, k in zip(ts, ks):

            bi = bp * t
            ci = cp * t

            # divisibility constraints
            if (k * bi) % Sj != 0:
                return

            if (k * ci) % Sj2 != 0:
                return

            ai = (k * bi) // Sj
            di = (k * ci) // Sj2

            a_list.append(ai)
            b_list.append(bi)
            c_list.append(ci)
            d_list.append(di)

        # Verify original equations exactly

        if sum(ai*bi for ai,bi in zip(a_list,b_list)) != a*b:
            return

        if sum(ci*di for ci,di in zip(c_list,d_list)) != c*d:
            return

        if sum(ai*ci for ai,ci in zip(a_list,c_list)) != a*c:
            return

        if sum(bi*di for bi,di in zip(b_list,d_list)) != b*d:
            return

        solutions.append({
            "a_i": a_list,
            "b_i": b_list,
            "c_i": c_list,
            "d_i": d_list,
            "k_i": ks,
        })

    def recurse(idx, remaining, ts, ks):

        if idx == n:
            if remaining == 0:
                build_solution(ts, ks)
            return

        terms_left = n - idx

        # Every remaining term contributes at least 1
        if remaining < terms_left:
            return

        # max_t = int(remaining**0.5)
        max_t = isqrt(remaining)

        for t in range(1, max_t + 1):

            t2 = t * t

            max_k = remaining // t2

            for k in range(1, max_k + 1):

                contrib = k * t2

                recurse(
                    idx + 1,
                    remaining - contrib,
                    ts + [t],
                    ks + [k]
                )

    recurse(0, M, [], [])

    return solutions

In [15]:
sols = search_solutions(
    a=3,
    b=2,
    c=2,
    d=3,
    Sj=4,
    Sj2=4,
    n=2
)

print(len(sols))
for s in sols[:10]:
    print(s)

15
{'a_i': [1, 5], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [1, 5], 'k_i': [4, 20]}
{'a_i': [2, 4], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [2, 4], 'k_i': [8, 16]}
{'a_i': [2, 2], 'b_i': [1, 2], 'c_i': [1, 2], 'd_i': [2, 2], 'k_i': [8, 4]}
{'a_i': [2, 1], 'b_i': [1, 4], 'c_i': [1, 4], 'd_i': [2, 1], 'k_i': [8, 1]}
{'a_i': [3, 3], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [3, 3], 'k_i': [12, 12]}
{'a_i': [4, 2], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [4, 2], 'k_i': [16, 8]}
{'a_i': [4, 1], 'b_i': [1, 2], 'c_i': [1, 2], 'd_i': [4, 1], 'k_i': [16, 2]}
{'a_i': [5, 1], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [5, 1], 'k_i': [20, 4]}
{'a_i': [1, 4], 'b_i': [2, 1], 'c_i': [2, 1], 'd_i': [1, 4], 'k_i': [2, 16]}
{'a_i': [1, 2], 'b_i': [2, 2], 'c_i': [2, 2], 'd_i': [1, 2], 'k_i': [2, 4]}


In [16]:
from functools import lru_cache
from math import isqrt

def decomposition_dp(M, n):

    @lru_cache(None)
    def solve(remaining, slots, min_t, min_k):

        if slots == 0:
            return [()] if remaining == 0 else []

        if remaining < slots:
            return []

        results = []

        max_t = isqrt(remaining)

        for t in range(min_t, max_t + 1):

            if t == min_t:
                start_k = min_k
            else:
                start_k = 1

            max_k = remaining // (t*t)

            for k in range(start_k, max_k + 1):

                contribution = k*t*t

                future_min = slots - 1

                if remaining - contribution < future_min:
                    break

                tails = solve(
                    remaining - contribution,
                    slots - 1,
                    t,
                    k
                )

                pair = (t,k)

                for tail in tails:
                    results.append((pair,) + tail)

        return results

    return solve(M, n, 1, 1)

In [17]:
decomposition_dp(24,2)

[((1, 1), (1, 23)),
 ((1, 2), (1, 22)),
 ((1, 3), (1, 21)),
 ((1, 4), (1, 20)),
 ((1, 4), (2, 5)),
 ((1, 5), (1, 19)),
 ((1, 6), (1, 18)),
 ((1, 6), (3, 2)),
 ((1, 7), (1, 17)),
 ((1, 8), (1, 16)),
 ((1, 8), (2, 4)),
 ((1, 8), (4, 1)),
 ((1, 9), (1, 15)),
 ((1, 10), (1, 14)),
 ((1, 11), (1, 13)),
 ((1, 12), (1, 12)),
 ((1, 12), (2, 3)),
 ((1, 15), (3, 1)),
 ((1, 16), (2, 2)),
 ((1, 20), (2, 1)),
 ((2, 1), (2, 5)),
 ((2, 2), (2, 4)),
 ((2, 2), (4, 1)),
 ((2, 3), (2, 3))]

In [18]:
from math import gcd

def recover_solution(solution, a, b, c, d, Sj, Sj2):
    """
    solution = ((t1,k1),...,(tn,kn))
    """

    g = gcd(b, c)
    bp = b // g
    cp = c // g

    a_list = []
    b_list = []
    c_list = []
    d_list = []

    for t, k in solution:

        bi = bp * t
        ci = cp * t

        ai = (k * bi) // Sj
        di = (k * ci) // Sj2

        a_list.append(ai)
        b_list.append(bi)
        c_list.append(ci)
        d_list.append(di)

    if sum(ai*bi for ai,bi in zip(a_list,b_list)) != a*b:
        return

    if sum(ci*di for ci,di in zip(c_list,d_list)) != c*d:
        return

    if sum(ai*ci for ai,ci in zip(a_list,c_list)) != a*c:
        return

    if sum(bi*di for bi,di in zip(b_list,d_list)) != b*d:
        return

    return {
        "a_i": a_list,
        "b_i": b_list,
        "c_i": c_list,
        "d_i": d_list,
        "k_i": [k for t, k in solution]
    }

In [19]:
a, b, c, d = 3, 2, 2, 3
Sj, Sj2 = 4, 4
g = gcd(b,c)
bp = b // g
M = (a*b*Sj) // (bp**2)
n=2

decomps = decomposition_dp(M,n)

solutions = []
for decomp in decomps:
    solution = recover_solution(decomp, a, b, c, d, Sj, Sj2)
    if solution is not None:
        print(solution)
        solutions.append(solution)

{'a_i': [1, 5], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [1, 5], 'k_i': [4, 20]}
{'a_i': [2, 4], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [2, 4], 'k_i': [8, 16]}
{'a_i': [2, 2], 'b_i': [1, 2], 'c_i': [1, 2], 'd_i': [2, 2], 'k_i': [8, 4]}
{'a_i': [2, 1], 'b_i': [1, 4], 'c_i': [1, 4], 'd_i': [2, 1], 'k_i': [8, 1]}
{'a_i': [3, 3], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [3, 3], 'k_i': [12, 12]}
{'a_i': [4, 1], 'b_i': [1, 2], 'c_i': [1, 2], 'd_i': [4, 1], 'k_i': [16, 2]}
{'a_i': [1, 2], 'b_i': [2, 2], 'c_i': [2, 2], 'd_i': [1, 2], 'k_i': [2, 4]}
{'a_i': [1, 1], 'b_i': [2, 4], 'c_i': [2, 4], 'd_i': [1, 1], 'k_i': [2, 1]}


In [20]:
a, b, c, d = 4,3,3,4
Sj, Sj2 = 15,15
g = gcd(b,c)
bp = b // g
M = (a*b*Sj) // (bp**2)
n=2

decomps = decomposition_dp(M,n)

solutions = []
for decomp in decomps:
    solution = recover_solution(decomp, a, b, c, d, Sj, Sj2)
    if solution is not None:
        print(solution)
        solutions.append(solution)

{'a_i': [1, 11], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [1, 11], 'k_i': [15, 165]}
{'a_i': [2, 10], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [2, 10], 'k_i': [30, 150]}
{'a_i': [2, 2], 'b_i': [1, 5], 'c_i': [1, 5], 'd_i': [2, 2], 'k_i': [30, 6]}
{'a_i': [3, 9], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [3, 9], 'k_i': [45, 135]}
{'a_i': [3, 3], 'b_i': [1, 3], 'c_i': [1, 3], 'd_i': [3, 3], 'k_i': [45, 15]}
{'a_i': [4, 8], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [4, 8], 'k_i': [60, 120]}
{'a_i': [4, 4], 'b_i': [1, 2], 'c_i': [1, 2], 'd_i': [4, 4], 'k_i': [60, 30]}
{'a_i': [5, 7], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [5, 7], 'k_i': [75, 105]}
{'a_i': [6, 6], 'b_i': [1, 1], 'c_i': [1, 1], 'd_i': [6, 6], 'k_i': [90, 90]}
{'a_i': [6, 2], 'b_i': [1, 3], 'c_i': [1, 3], 'd_i': [6, 2], 'k_i': [90, 10]}
{'a_i': [7, 1], 'b_i': [1, 5], 'c_i': [1, 5], 'd_i': [7, 1], 'k_i': [105, 3]}
{'a_i': [8, 2], 'b_i': [1, 2], 'c_i': [1, 2], 'd_i': [8, 2], 'k_i': [120, 15]}
{'a_i': [9, 1], 'b_i': [1, 3], 'c_i': [1, 3], 'd_i': [9

### General Dynamic Programming Algorithm

In [21]:
def combinations_leq_m(m, n, minimum=1):
    if n == 0:
        yield []
        return

    # Need at least n * minimum remaining
    if m < n * minimum:
        return

    max_first = m // n

    for first in range(minimum, max_first + 1):
        for rest in combinations_leq_m(
            m - first,
            n - 1,
            first
        ):
            yield [first,] + rest

In [ ]:
def initial_step_ep(p, q, SI, k_max, num_paths):
    path_total = int(p * q)
    possible_p_combinations = list(combinations_leq_m(path_total, num_paths))

    solutions = []
    
    for P in possible_p_combinations:
        possible_q_combinations = []
        for p_i in P:
            divisor_num = int(SI * p_i)
            possible_q_combinations.append([d for d in divisors(divisor_num) if d < SI])

        # Implement branch and bound for optimal searching of viable p_i, q_i
        min_contrib = [0]*(num_paths+1)
        max_contrib = [0]*(num_paths+1)

        for i in range(num_paths-1, -1, -1):
            vals = [P[i]*q for q in possible_q_combinations[i]]

            min_contrib[i] = min_contrib[i+1] + min(vals)
            max_contrib[i] = max_contrib[i+1] + max(vals)

        def recurse(i, current_sum, chosen):

            if i == num_paths:
                if current_sum == path_total:
                    yield list(chosen)
                return

            # prune impossible branches
            if current_sum + min_contrib[i] > path_total:
                return

            if current_sum + max_contrib[i] < path_total:
                return

            p = P[i]

            for q in possible_q_combinations[i]:
                # Enforce ordering among equal p's
                if i > 0 and P[i] == P[i-1]:
                    if q < chosen[-1]:
                        continue
                yield from recurse(
                    i+1,
                    current_sum + p*q,
                    chosen + [q]
                )
        
        for Q in recurse(0, 0, []):
            solutions.append([list(edge) for edge in zip(P,Q)])

    ks = [[0 for _ in range(num_paths)] for _ in range(len(solutions))]
    path_product_weights = [[0 for _ in range(num_paths)] for _ in range(len(solutions))]
    for i, solution in enumerate(solutions):
        for j, path in enumerate(solution):
            p_i = path[0]
            q_i = path[1]
            ks[i][j] = [SI*p_i/q_i]
            path_product_weights[i][j] = [p_i * q_i]

    return solutions, ks, path_product_weights

solutions, ks, path_product_weights = initial_step_ep(3,2,6,2)
print(solutions)
print(ks)
print(path_product_weights)
path_product_weights[1]

[[[1, 3], [1, 3]], [[1, 2], [2, 2]], [[1, 3], [3, 1]], [[1, 2], [4, 1]], [[1, 1], [5, 1]], [[2, 1], [2, 2]], [[2, 1], [4, 1]], [[3, 1], [3, 1]]]
[[[2.0], [2.0]], [[3.0], [6.0]], [[2.0], [18.0]], [[3.0], [24.0]], [[6.0], [30.0]], [[12.0], [6.0]], [[12.0], [24.0]], [[18.0], [18.0]]]
[[[3], [3]], [[2], [4]], [[3], [3]], [[2], [4]], [[1], [5]], [[2], [4]], [[2], [4]], [[3], [3]]]


[[2], [4]]

In [ ]:
from functools import lru_cache

def weighted_solutions(weights, M):

    n = len(weights)
    min_future = [0]*(n+1)

    for i in range(n-1, -1, -1):
        min_future[i] = min_future[i+1] + weights[i][-1]

    @lru_cache(None)
    def solve(i, remaining):
        if i == n:
            return [[]] if remaining == 0 else []

        w = weights[i][-1]
        future_min = min_future[i+1]
        max_r = (remaining - future_min) // w
        results = []

        for r in range(1, max_r + 1):

            tails = solve(
                i + 1,
                remaining - w*r
            )

            for tail in tails:
                results.append([r,] + tail)

        return results

    return solve(0, M)

weighted_solutions([[2],[4],[3]], 16)

[[1, 2, 2], [3, 1, 2]]

In [ ]:
from sympy import divisors

def admissible_pairs(r, k):

    pairs = []
    for p in divisors(r):
        q = r // p

        if (k*p) % q == 0 and q <= k:
            pairs.append([p,q])

    return pairs

In [ ]:
def intermediate_step_ep(m, ks, solutions, path_product_weights, current_path_product, SI, num_paths):
    new_solutions = []
    for i, solution in enumerate(solutions):
        possible_rs_combinations = weighted_solutions(path_product_weights[i], current_path_product)
        for rs in possible_rs_combinations:
            new_weighted_paths = [[] for _ in range(num_paths)]
            for j, r in enumerate(rs):
                k = ks[i][j][m]
                pairs = admissible_pairs(r, k)
                if len(pairs) == 0:
                    continue
                for pair in pairs:
                    new_weighted_paths[j].append(solutions[i][j].copy())
                    new_weighted_paths[j][-1].extend(pair)
            weighted_paths_combinations = itertools.product(*new_weighted_paths)
            new_solutions.extend([list(new_solution) for new_solution in weighted_paths_combinations])
        
    new_ks = [[[] for _ in range(num_paths)] for _ in range(len(new_solutions))]
    new_path_product_weights = [[[] for _ in range(num_paths)] for _ in range(len(new_solutions))]
    for i, solution in enumerate(new_solutions):
        for j, path in enumerate(solution):
            for l in range(0,len(path),2):
                p_i = path[l]
                q_i = path[l+1]
                if l == 0:
                    k_i = SI * p_i / q_i
                    w_i = p_i * q_i
                else:
                    k_i = new_ks[i][j][-1] * p_i / q_i
                    w_i = new_path_product_weights[i][j][-1] * p_i * q_i
                new_ks[i][j].append(k_i)
                new_path_product_weights[i][j].append(w_i)

    return new_solutions, new_ks, new_path_product_weights
            

In [38]:
def prune_solutions(solutions, ks, path_product_weights, ks_max):
    print(ks)
    print(solutions)

    pruned_solutions = []
    pruned_ks = []
    pruned_path_product_weights = []

    for i, solution in enumerate(solutions):
        below_max = True
        for j, path in enumerate(solution):
            for l in range(len(path) // 2):
                if ks[i][j][l] > ks_max[l]:
                    below_max = False
                    break
            if not below_max:
                break
        if below_max:
            pruned_solutions.append(solution)
            pruned_ks.append(ks[i])
            pruned_path_product_weights.append(path_product_weights[i])

    return pruned_solutions, pruned_ks, pruned_path_product_weights
        

In [ ]:
def general_weighted_path_unfolding(ps, qs, SI, SF, ks_max, num_paths):
    """Compute set of valid equitable partition unfoldings of a weighted 
    path that maintain the same number of non-returning walks of all lengths.

    ps: array(int)
        List of integers whose i-th entry represents the number of edges from partition i-1 to partition i
    qs: array(int)
        List of integers whose i-th entry represents the number of edges from partition i to partition i-1
    SI: int
        Number of vertices in the initial partition
    SF: int
        Number of vertices in the final partition
    num_paths:  int
        Number of paths to separate the weighted path into
    """

    path_len = len(ps)
    if path_len != len(qs):
        print("Length of ps does not match qs")
        return

    # Initial step and separation
    solutions, ks, path_product_weights = initial_step_ep(ps[0], qs[0], SI, num_paths)

    # Prune to manageable amount
    solutions, ks, path_product_weights = prune_solutions(solutions, ks, path_product_weights, ks_max)

    # Intermediate steps
    current_path_product = ps[0] * qs[0]
    for i in range(1,path_len-1):
        current_path_product *= ps[i] * qs[i]
        m = i - 1
        solutions, ks, path_product_weights = intermediate_step_ep(m, ks, solutions, path_product_weights, current_path_product, SI, num_paths)
        solutions, ks, path_product_weights = prune_solutions(solutions, ks, path_product_weights, ks_max)

# ps = [4,3,2]
# qs = [2,3,4]
# SI = 5
# SF = 5
# num_paths = 2

ps = [5,4,3,2]
qs = [2,3,4,5]
SI = 6
SF = 6
ks_max = [10,10,10,10]
num_paths = 3

general_weighted_path_unfolding(ps, qs, SI, SF, ks_max, num_paths)

[[[6.0], [6.0], [3.0]], [[6.0], [2.0], [4.0]], [[3.0], [3.0], [4.0]], [[2.0], [2.0], [6.0]], [[6.0], [2.0], [9.0]], [[3.0], [3.0], [9.0]], [[6.0], [6.0], [12.0]], [[2.0], [2.0], [24.0]], [[3.0], [2.0], [30.0]], [[6.0], [2.0], [36.0]], [[3.0], [3.0], [36.0]], [[6.0], [3.0], [42.0]], [[6.0], [6.0], [48.0]], [[3.0], [12.0], [4.0]], [[3.0], [6.0], [6.0]], [[6.0], [4.0], [18.0]], [[3.0], [12.0], [9.0]], [[2.0], [6.0], [18.0]], [[3.0], [6.0], [24.0]], [[6.0], [6.0], [30.0]], [[2.0], [12.0], [30.0]], [[3.0], [12.0], [36.0]], [[6.0], [12.0], [42.0]], [[6.0], [18.0], [9.0]], [[2.0], [18.0], [24.0]], [[3.0], [18.0], [30.0]], [[6.0], [18.0], [36.0]], [[3.0], [24.0], [24.0]], [[6.0], [24.0], [30.0]], [[12.0], [12.0], [4.0]], [[12.0], [6.0], [6.0]], [[12.0], [12.0], [9.0]], [[12.0], [6.0], [24.0]], [[12.0], [12.0], [36.0]], [[6.0], [18.0], [18.0]], [[12.0], [18.0], [30.0]], [[12.0], [24.0], [24.0]], [[18.0], [18.0], [24.0]]]
[[[1, 1], [1, 1], [2, 4]], [[1, 1], [1, 3], [2, 3]], [[1, 2], [1, 2], [2, 

In [ ]:
test = [1,2,3,4]
print(len(test))
list(range(0,4,2))

4


[0, 2]